# PR 5 — Sentiment Analysis with RNN, LSTM and GRU

Dataset: IMDB Dataset of 50K Movie Reviews. Put `IMDB Dataset.csv` in the same folder as this notebook before running it.

This notebook is split into separate cells for each practical step. Run from top to bottom.

## Setup

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_fscore_support
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, SimpleRNN, LSTM, GRU, RepeatVector, Lambda
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from IPython.display import display

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)
warnings.filterwarnings('ignore')
sns.set_theme()

## Task 1 — Text loading, cleaning, tokenisation and padding

### 1.1 Load and inspect

In [ ]:
DATA_PATH = 'IMDB Dataset.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())
print('\nSentiment counts:')
print(df['sentiment'].value_counts())
print('\nMissing values:')
print(df.isnull().sum())
print('\nTwo raw reviews:')
print(df.loc[0, 'review'])
print('\n--- Second review ---\n')
print(df.loc[1, 'review'])

### 1.2 Remove duplicates

In [ ]:
rows_before = len(df)
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print('Duplicates:', duplicate_count)
print('Rows before:', rows_before)
print('Rows after:', len(df))

Duplicates are removed before splitting so the same review cannot leak into both training and test data.

### 1.3 Clean the text

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'<.*?>', ' ', s)
    s = re.sub(r"[^a-z']", ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

print('Before:\n', df.loc[0, 'review'])
print('\nAfter:\n', clean_text(df.loc[0, 'review']))
df['clean_review'] = df['review'].apply(clean_text)
df['y'] = df['sentiment'].map({'negative': 0, 'positive': 1})

Apostrophes are kept because contractions carry meaning. Stop-words stay because words such as “not” and “no” can reverse sentiment.

### 1.4 Review length analysis

In [ ]:
df['word_count'] = df['clean_review'].str.split().str.len()
mean_len = df['word_count'].mean()
median_len = df['word_count'].median()
p95_len = df['word_count'].quantile(0.95)
print('Mean:', round(mean_len, 2), 'Median:', median_len, '95th percentile:', p95_len)

plt.figure(figsize=(10, 5))
plt.hist(df['word_count'], bins=50)
plt.axvline(mean_len, linestyle='--', label=f'Mean {mean_len:.1f}')
plt.axvline(median_len, linestyle='--', label=f'Median {median_len:.1f}')
plt.axvline(p95_len, linestyle='--', label=f'95th percentile {p95_len:.1f}')
plt.title('Review Length Distribution')
plt.xlabel('Words per review')
plt.ylabel('Number of reviews')
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='sentiment', y='word_count')
plt.title('Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Word count')
plt.show()

### 1.5 Split first, then fit tokenizer

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_review'], df['y'], test_size=0.2, random_state=42, stratify=df['y']
)
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)
train_sequences = tokenizer.texts_to_sequences(X_train_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)
print('Top 10 words:', list(tokenizer.word_index.items())[:10])
print('One review sequence:', train_sequences[0][:50])

Only training text is used to fit the tokenizer to avoid test-data leakage. `num_words=10000` limits rare-word noise; OOV handles unseen words.

### 1.6 Pad and truncate

In [ ]:
MAXLEN = int(round(np.percentile(df['word_count'], 90) / 10) * 10)
MAXLEN = max(50, MAXLEN)
MAXLEN = min(MAXLEN, 300)
X_train = pad_sequences(train_sequences, maxlen=MAXLEN, padding='pre', truncating='pre')
X_test = pad_sequences(test_sequences, maxlen=MAXLEN, padding='pre', truncating='pre')
y_train = np.asarray(y_train, dtype='int32')
y_test = np.asarray(y_test, dtype='int32')
print('MAXLEN:', MAXLEN)
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_test:', X_test.shape, 'y_test:', y_test.shape)

Pre-padding keeps real words at the end, which is useful when the final hidden state is used for classification.

### 1.7 Why text is sequential

Word order changes meaning: “the movie was not good” differs from “the movie was good, not bad”. A bag-of-words model can ignore this order, while recurrent networks process tokens in sequence.

## Task 2 — RNN intuition, ANN baseline and forward propagation

### 2.1 Order-blind ANN baseline

In [ ]:
ann_model = Sequential([
    Input(shape=(MAXLEN,)),
    Embedding(10000, 32),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann_model.summary()
ann_start = time.time()
ann_history = ann_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=1)
ann_seconds = (time.time() - ann_start) / max(1, len(ann_history.epoch))
ann_prob = ann_model.predict(X_test, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)
ann_acc = accuracy_score(y_test, ann_pred)
print('ANN test accuracy:', ann_acc)
print('ANN parameters:', ann_model.count_params(), 'seconds/epoch:', ann_seconds)

An ANN with Flatten treats each position as a separate slot. Its dense parameter count grows with sequence length and it does not naturally share word-processing weights across positions.

### 2.2 RNN intuition

An RNN reads one token at a time and carries a hidden state. The same weights are reused at every time step.

\(h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)\)

\(y_t = f(W_{hy}h_t + b_y)\)

Here \(x_t\) is the current input, \(h_{t-1}\) is the previous hidden state, and \(h_t\) summarises information read so far.

### 2.3 Manual RNN forward pass in NumPy

In [ ]:
np.random.seed(42)
input_size, hidden_size, steps = 2, 3, 4
Wxh = np.random.randn(hidden_size, input_size) * 0.5
Whh = np.random.randn(hidden_size, hidden_size) * 0.5
b = np.random.randn(hidden_size) * 0.5
sequence = np.random.randn(steps, input_size)
h = np.zeros(hidden_size)
manual_states = []
for t in range(steps):
    h = np.tanh(Wxh @ sequence[t] + Whh @ h + b)
    manual_states.append(h.copy())
    print(f'h({t+1}) =', h)
manual_states = np.array(manual_states)
plt.figure(figsize=(6, 3))
sns.heatmap(manual_states.T, annot=True, cmap='coolwarm', xticklabels=[1,2,3,4], yticklabels=['Unit 1','Unit 2','Unit 3'])
plt.title('Manual RNN Forward-Pass Heatmap')
plt.xlabel('Time step')
plt.ylabel('Hidden unit')
plt.show()

### 2.4 Verify against Keras SimpleRNN

In [ ]:
rnn_check = Sequential([Input(shape=(steps, input_size)), SimpleRNN(hidden_size, return_sequences=True, activation='tanh')])
rnn_check(np.zeros((1, steps, input_size)))
rnn_check.layers[0].set_weights([Wxh.T, Whh.T, b])
keras_states = rnn_check.predict(sequence[np.newaxis, :, :], verbose=0)[0]
print('Keras equals manual:', np.allclose(keras_states, manual_states, atol=1e-6))
print('Keras states:\n', keras_states)

Keras stores the input kernel as input-size × units, so the NumPy input matrix is transposed when setting weights.

### 2.5 SimpleRNN parameter count

In [ ]:
units, embedding_dim = 64, 32
print('Formula:', units, '* (', embedding_dim, '+', units, '+ 1) =', units * (embedding_dim + units + 1))

For 64 units and 32 input features, the count is 6,208: input weights + recurrent weights + bias. It does not depend on MAXLEN.

### 2.6 ANN vs RNN

| Feature | ANN baseline | RNN |
|---|---|---|
| Input handling | Fixed positions | One time step at a time |
| Word order | Not naturally modelled | Uses sequence order |
| Weight sharing | No across positions | Same weights across time |
| Parameters vs length | Flatten-based dense parameters grow with length | Recurrent parameters stay fixed |
| Earlier words | No recurrent state | Summarised in hidden state |
| Typical use | Tabular/fixed features | Text, speech, time series |

## Task 3 — RNN types and SimpleRNN sentiment model

### 3.1 Demonstrate four RNN types

In [ ]:
dummy = np.random.randn(4, 10, 8).astype('float32')
one_to_one = Dense(1)
print('One-to-One:', one_to_one(dummy[:, 0, :]).shape)
one_to_many = Sequential([Input(shape=(8,)), RepeatVector(10), SimpleRNN(16, return_sequences=True)])
print('One-to-Many:', one_to_many(np.random.randn(4, 8).astype('float32')).shape)
many_to_one = SimpleRNN(16)
print('Many-to-One:', many_to_one(dummy).shape)
many_to_many = SimpleRNN(16, return_sequences=True)
print('Many-to-Many:', many_to_many(dummy).shape)

| Type | Example | Project use? |
|---|---|---|
| One-to-One | Fixed input to label | ANN baseline |
| One-to-Many | Image caption generation | No |
| Many-to-One | Sentiment classification | Yes: one review → one label |
| Many-to-Many | Token tagging or translation | No |

### 3.2 Why sentiment is many-to-one

A complete review is the input sequence and its sentiment is a single output label, so this project uses many-to-one classification.

### 3.3 Build model function

In [ ]:
def build_model(cell_type='rnn', units=64, maxlen=MAXLEN, learning_rate=0.001):
    model = Sequential()
    model.add(Input(shape=(maxlen,)))
    model.add(Embedding(10000, 32))
    if cell_type == 'rnn':
        model.add(SimpleRNN(units))
    elif cell_type == 'lstm':
        model.add(LSTM(units))
    elif cell_type == 'gru':
        model.add(GRU(units))
    else:
        raise ValueError('cell_type must be rnn, lstm, or gru')
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model

### 3.4 Train SimpleRNN

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
def train_model(model, Xtr=X_train, ytr=y_train, epochs=15):
    start = time.time()
    hist = model.fit(Xtr, ytr, epochs=epochs, batch_size=64, validation_split=0.1,
                     callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)], verbose=1)
    seconds_per_epoch = (time.time() - start) / max(1, len(hist.epoch))
    return hist, seconds_per_epoch

simple_rnn = build_model('rnn', 64)
simple_rnn.summary()
rnn_history, rnn_seconds = train_model(simple_rnn)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='Train')
    axes[0].plot(history.history['val_loss'], label='Validation')
    axes[0].set_title(title + ' Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[1].plot(history.history['accuracy'], label='Train')
    axes[1].plot(history.history['val_accuracy'], label='Validation')
    axes[1].set_title(title + ' Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
plot_history(rnn_history, 'SimpleRNN')

### 3.5 Evaluate models

In [ ]:
results = {}
probabilities = {}
histories = {}
models = {}
def evaluate_model(name, model, history, seconds_per_epoch):
    prob = model.predict(X_test, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    report = classification_report(y_test, pred, output_dict=True, zero_division=0)
    metrics = {
        'Model': name, 'Parameters': model.count_params(), 'Time per Epoch (s)': seconds_per_epoch,
        'Epochs Trained': len(history.epoch), 'Test Accuracy': accuracy_score(y_test, pred),
        'Precision': report['1']['precision'], 'Recall': report['1']['recall'],
        'F1': report['1']['f1-score'], 'ROC-AUC': roc_auc_score(y_test, prob)
    }
    results[name] = metrics
    probabilities[name] = prob
    histories[name] = history
    models[name] = model
    print(name, classification_report(y_test, pred, zero_division=0))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues')
    plt.title(name + ' Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    return metrics

evaluate_model('SimpleRNN', simple_rnn, rnn_history, rnn_seconds)

### 3.6 Embedding layer

In [ ]:
embedding_weights = simple_rnn.layers[0].get_weights()[0]
print('Embedding weights shape:', embedding_weights.shape)
for word in ['good', 'bad']:
    word_id = tokenizer.word_index.get(word)
    if word_id is not None and word_id < 10000:
        print(word, embedding_weights[word_id][:10])

Embeddings map integer word IDs to learned dense vectors. They are trained from scratch in this practical.

## Task 4 — BPTT and RNN problems

### 4.1 Backpropagation Through Time

BPTT unrolls the RNN over time. The final loss sends gradients backwards through each step. The gradient to an early state contains a product of repeated Jacobian terms, while each shared weight receives contributions from every time step.

\(\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_T}\prod_{t=2}^{T}(W_{hh}^{T}\,\mathrm{diag}(1-h_t^2))\)

### 4.2 Vanishing and exploding gradients

If repeated factors are smaller than 1, gradients can shrink exponentially (vanishing). If they grow above 1, gradients can explode and cause unstable loss or NaNs. SimpleRNNs can struggle with long-range dependencies.

### 4.3 GradientTape per-time-step gradient norms

In [ ]:
gradient_model = Sequential([
    Input(shape=(MAXLEN,)),
    Embedding(10000, 32),
    SimpleRNN(64, return_sequences=True),
    Lambda(lambda h: h[:, -1, :]),
    Dense(1, activation='sigmoid')
])
gradient_model.compile(optimizer='adam', loss='binary_crossentropy')
batch_x = tf.convert_to_tensor(X_train[:64])
batch_y = tf.convert_to_tensor(y_train[:64].reshape(-1, 1), dtype=tf.float32)
with tf.GradientTape() as tape:
    emb = gradient_model.layers[0](batch_x)
    H = gradient_model.layers[1](emb)
    tape.watch(H)
    final_h = H[:, -1, :]
    pred = gradient_model.layers[-1](final_h)
    loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(batch_y, pred))
grads = tape.gradient(loss, H)
grad_norm = tf.reduce_mean(tf.norm(grads, axis=-1), axis=0).numpy()
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, MAXLEN + 1), grad_norm)
plt.yscale('log')
plt.title('Gradient Norm per Time Step — SimpleRNN')
plt.xlabel('Time step')
plt.ylabel('Gradient norm (log scale)')
plt.show()

### 4.4 Accuracy vs sequence length

In [ ]:
length_results = []
for length in [50, 100, 200, 300]:
    xtr_len = pad_sequences(train_sequences, maxlen=length, padding='pre', truncating='pre')
    xte_len = pad_sequences(test_sequences, maxlen=length, padding='pre', truncating='pre')
    model_len = build_model('rnn', 64, maxlen=length)
    hist_len, sec_len = train_model(model_len, xtr_len, y_train, epochs=8)
    pred_len = model_len.predict(xte_len, verbose=0).ravel()
    length_results.append({'Sequence length': length, 'Test accuracy': accuracy_score(y_test, pred_len >= 0.5)})
    del model_len
length_df = pd.DataFrame(length_results)
display(length_df)
plt.plot(length_df['Sequence length'], length_df['Test accuracy'], marker='o')
plt.title('SimpleRNN Accuracy vs Sequence Length')
plt.xlabel('Sequence length')
plt.ylabel('Test accuracy')
plt.show()

### 4.5 Exploding gradient demo and clipping

In [ ]:
def train_high_lr(clipnorm=None):
    model = build_model('rnn', 32, learning_rate=0.05)
    model.compile(optimizer=Adam(learning_rate=0.05, clipnorm=clipnorm), loss='binary_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=1)
    return hist

high_lr_history = train_high_lr()
clipped_history = train_high_lr(clipnorm=1.0)
plt.figure(figsize=(8, 4))
plt.plot(high_lr_history.history['loss'], label='High learning rate')
plt.plot(clipped_history.history['loss'], label='With clipnorm=1.0')
plt.title('Exploding Gradient Demo')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

| Problem | Cause | What to inspect | Remedy |
|---|---|---|---|
| Vanishing gradient | Repeated small derivatives | Early-step gradient norms | LSTM/GRU gates |
| Exploding gradient | Repeated large factors | Loss spikes or NaNs | Gradient clipping |
| Slow sequential training | Time steps are processed in order | Epoch time | Shorter sequences or GPU |
| Short memory | Weak long-range signal | Long-review performance | Gated cells |

## Task 5 — LSTM

### 5.1 LSTM theory

LSTM uses a cell state \(C_t\), forget gate \(f_t\), input gate \(i_t\), candidate \(g_t\), and output gate \(o_t\).

\(f_t=\sigma(W_f[h_{t-1},x_t]+b_f)\), \(i_t=\sigma(W_i[h_{t-1},x_t]+b_i)\)

\(g_t=\tanh(W_g[h_{t-1},x_t]+b_g)\), \(C_t=f_t*C_{t-1}+i_t*g_t\)

\(o_t=\sigma(W_o[h_{t-1},x_t]+b_o)\), \(h_t=o_t*\tanh(C_t)\)

The forget gate chooses what to retain, the input gate chooses what to write, and the output gate chooses what to reveal.

### 5.2 Why LSTM helps with vanishing gradients

The cell state has an additive update path. When the forget gate stays near 1, information and gradients can pass through many steps with less shrinkage.

### 5.3 LSTM parameter count

In [ ]:
print('LSTM formula:', 4 * units * (embedding_dim + units + 1))
lstm_check = build_model('lstm', 64)
print('Keras count:', lstm_check.count_params())

### 5.4 Manual single-step LSTM in NumPy

In [ ]:
np.random.seed(42)
input_size, hidden_size = 2, 3
W = np.random.randn(4 * hidden_size, input_size + hidden_size) * 0.2
bias = np.random.randn(4 * hidden_size) * 0.1
x_seq = np.random.randn(4, input_size)
h = np.zeros(hidden_size)
c = np.zeros(hidden_size)
def sigmoid(x): return 1 / (1 + np.exp(-x))
for t, x in enumerate(x_seq, start=1):
    gates = W @ np.concatenate([h, x]) + bias
    f, i, g, o = np.split(gates, 4)
    f, i, o = sigmoid(f), sigmoid(i), sigmoid(o)
    g = np.tanh(g)
    c = f * c + i * g
    h = o * np.tanh(c)
    print('step', t, 'f=', np.round(f, 3), 'i=', np.round(i, 3), 'o=', np.round(o, 3), 'C=', np.round(c, 3), 'h=', np.round(h, 3))

When the forget gate is close to 1 and input gate close to 0, the cell state is mostly preserved.

### 5.5 Train and evaluate LSTM

In [ ]:
lstm_model = build_model('lstm', 64)
lstm_history, lstm_seconds = train_model(lstm_model)
plot_history(lstm_history, 'LSTM')
evaluate_model('LSTM', lstm_model, lstm_history, lstm_seconds)

### 5.6 Compare gradient flow

In [ ]:
def gradient_norms_for_cell(cell_type):
    inp = Input(shape=(MAXLEN,))
    x = Embedding(10000, 32)(inp)
    layer = {'rnn': SimpleRNN, 'lstm': LSTM, 'gru': GRU}[cell_type](64, return_sequences=True)
    H = layer(x)
    last = Lambda(lambda h: h[:, -1, :])(H)
    out = Dense(1, activation='sigmoid')(last)
    m = Model(inp, out)
    xb = tf.convert_to_tensor(X_train[:64])
    yb = tf.convert_to_tensor(y_train[:64].reshape(-1, 1), dtype=tf.float32)
    with tf.GradientTape() as tape:
        emb = m.layers[1](xb)
        H_tensor = m.layers[2](emb)
        tape.watch(H_tensor)
        pred = m.layers[-1](H_tensor[:, -1, :])
        loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb, pred))
    g = tape.gradient(loss, H_tensor)
    return tf.reduce_mean(tf.norm(g, axis=-1), axis=0).numpy()

rnn_grad_norm = grad_norm
lstm_grad_norm = gradient_norms_for_cell('lstm')
gru_grad_norm = gradient_norms_for_cell('gru')
plt.figure(figsize=(9, 5))
for label, vals in [('SimpleRNN', rnn_grad_norm), ('LSTM', lstm_grad_norm), ('GRU', gru_grad_norm)]:
    plt.plot(np.arange(1, len(vals)+1), vals, label=label)
plt.yscale('log')
plt.title('Gradient Norm Comparison')
plt.xlabel('Time step')
plt.ylabel('Gradient norm (log scale)')
plt.legend()
plt.show()

### 5.7 Stacked LSTM

In [ ]:
stacked_lstm = Sequential([
    Input(shape=(MAXLEN,)),
    Embedding(10000, 32),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
stacked_lstm.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
stacked_history, stacked_seconds = train_model(stacked_lstm)
plot_history(stacked_history, 'Stacked LSTM')
evaluate_model('Stacked LSTM', stacked_lstm, stacked_history, stacked_seconds)

The first LSTM returns the full sequence because the second LSTM needs a sequence, not only the first layer's final state.

## Task 6 — GRU

### 6.1 GRU theory

A GRU has an update gate \(z_t\), reset gate \(r_t\), and candidate state \(\tilde h_t\). It does not have a separate cell state.

\(z_t=\sigma(W_z[h_{t-1},x_t])\), \(r_t=\sigma(W_r[h_{t-1},x_t])\)

\(\tilde h_t=\tanh(W[r_t*h_{t-1},x_t])\)

\(h_t=(1-z_t)*h_{t-1}+z_t*\tilde h_t\)

The update gate balances old and new information; the reset gate controls how much past information enters the candidate.

| Feature | LSTM | GRU |
|---|---|---|
| Gates | 3 gates plus candidate | Update and reset gates plus candidate |
| Separate cell state | Yes | No |
| Parameter count | Usually higher | Usually lower |
| Speed | Often slower | Often faster |
| Use | Longer/complex dependencies | Efficient baseline and many sequence tasks |

### 6.2 GRU vs LSTM

The GRU combines the forget/input behaviour into one update gate, so it has fewer parameters than an LSTM with the same hidden size.

### 6.3 GRU parameter count

In [ ]:
gru_check = build_model('gru', 64)
print('GRU Keras parameters:', gru_check.count_params())
print('GRU recurrent layer parameters:', gru_check.layers[2].count_params())
print('Expected recurrent parameters:', 3 * (64 * 32 + 64 * 64 + 2 * 64))

### 6.4 Train and evaluate GRU

In [ ]:
gru_model = build_model('gru', 64)
gru_history, gru_seconds = train_model(gru_model)
plot_history(gru_history, 'GRU')
evaluate_model('GRU', gru_model, gru_history, gru_seconds)

### 6.5 Gradient flow — all three cells

Use the combined gradient comparison plot from Task 5.6 to compare how well gradients reach earlier time steps.

### 6.6 Hidden-unit sensitivity

In [ ]:
sensitivity = []
sensitivity_models = {}
for cell in ['lstm', 'gru']:
    for unit_count in [32, 64, 128]:
        m = build_model(cell, unit_count)
        h, sec = train_model(m, epochs=6)
        p = m.predict(X_test, verbose=0).ravel()
        sensitivity.append({'Cell': cell.upper(), 'Units': unit_count, 'Accuracy': accuracy_score(y_test, p >= 0.5), 'Parameters': m.count_params()})
        sensitivity_models[(cell, unit_count)] = m
sensitivity_df = pd.DataFrame(sensitivity)
display(sensitivity_df)
for metric in ['Accuracy', 'Parameters']:
    plt.figure(figsize=(7, 4))
    sns.lineplot(data=sensitivity_df, x='Units', y=metric, hue='Cell', marker='o')
    plt.title(f'{metric} vs Hidden Units')
    plt.show()

## Task 7 — Comparison, inference and recommendation

### 7.1 Full results table

In [ ]:
if 'ANN baseline' not in results:
    ann_report = classification_report(y_test, ann_pred, output_dict=True, zero_division=0)
    results['ANN baseline'] = {
        'Model': 'ANN baseline', 'Parameters': ann_model.count_params(), 'Time per Epoch (s)': ann_seconds,
        'Epochs Trained': len(ann_history.epoch), 'Test Accuracy': accuracy_score(y_test, ann_pred),
        'Precision': ann_report['1']['precision'], 'Recall': ann_report['1']['recall'],
        'F1': ann_report['1']['f1-score'], 'ROC-AUC': roc_auc_score(y_test, ann_prob)
    }
results_df = pd.DataFrame(results.values())
display(results_df.style.highlight_max(subset=['Test Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'], color='#ffcccc'))
os.makedirs('plots', exist_ok=True)
results_df.to_csv('plots/results_table.csv', index=False)

### 7.2 ROC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
fpr, tpr, _ = roc_curve(y_test, ann_prob)
plt.plot(fpr, tpr, label=f'ANN baseline (AUC={roc_auc_score(y_test, ann_prob):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

ROC curves show the trade-off between true-positive and false-positive rates across thresholds, while accuracy reports only one threshold.

### 7.3 Accuracy by review length

In [ ]:
best_name = max(results, key=lambda k: results[k]['F1'])
best_model = models.get(best_name, ann_model)
best_prob = probabilities.get(best_name, ann_prob)
test_indices = X_test_text.index
test_lengths = df.loc[test_indices, 'word_count'].to_numpy()
bins = [('Short', test_lengths < 100), ('Medium', (test_lengths >= 100) & (test_lengths <= 250)), ('Long', test_lengths > 250)]
length_eval = []
for model_name, model_obj in [('SimpleRNN', simple_rnn), (best_name, best_model)]:
    probs = model_obj.predict(X_test, verbose=0).ravel() if model_name != 'ANN baseline' else ann_prob
    preds = probs >= 0.5
    for label, mask in bins:
        if mask.sum():
            length_eval.append({'Model': model_name, 'Review length': label, 'Accuracy': accuracy_score(y_test[mask], preds[mask])})
length_eval_df = pd.DataFrame(length_eval).drop_duplicates()
display(length_eval_df)
sns.barplot(data=length_eval_df, x='Review length', y='Accuracy', hue='Model')
plt.title('Accuracy by Review Length')
plt.show()

### 7.4 Error analysis

In [ ]:
error_model_name = best_name
error_model = best_model
error_prob = error_model.predict(X_test, verbose=0).ravel() if error_model_name != 'ANN baseline' else ann_prob
error_pred = (error_prob >= 0.5).astype(int)
test_review_text = X_test_text.reset_index(drop=True)
error_rows = []
for i, (true, pred) in enumerate(zip(y_test, error_pred)):
    if true != pred:
        error_rows.append({
            'True label': 'positive' if true == 1 else 'negative',
            'Predicted label': 'positive' if pred == 1 else 'negative',
            'Predicted probability': float(error_prob[i]),
            'Cleaned text': test_review_text.iloc[i],
            'Type': 'False Positive' if pred == 1 else 'False Negative'
        })
error_df = pd.DataFrame(error_rows)
display(error_df[error_df['Type'] == 'False Positive'].head(4))
display(error_df[error_df['Type'] == 'False Negative'].head(4))

Look for sarcasm, negation, mixed opinions, long reviews and cases where a negative character appears in a positive film.

### 7.5 Inference function

In [ ]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAXLEN, padding='pre', truncating='pre')
    prob = float(best_model.predict(padded, verbose=0)[0][0])
    return {'text': text, 'prediction': 'positive' if prob >= 0.5 else 'negative', 'probability': round(prob, 4)}

custom_sentences = [
    'This movie was wonderful and I loved every minute.',
    'This film was boring and badly acted.',
    'This film is not good.',
    'I cannot say it was not enjoyable.',
    'Wow, what a masterpiece of terrible writing.',
    'Good.'
]
inference_df = pd.DataFrame([predict_sentiment(s) for s in custom_sentences])
display(inference_df)

### 7.6 Which Recurrent Cell Should We Use?

In [ ]:
display(results_df.sort_values('F1', ascending=False))

Use the measured results above to choose a model rather than assuming one cell always wins. Compare F1 and ROC-AUC alongside accuracy, then consider epoch time, parameter count and long-review behaviour. Limitations include training embeddings from scratch without pretrained language knowledge and reading the review in only one direction.

### 7.7 Save plots and notebook outputs

In [ ]:
os.makedirs('plots', exist_ok=True)
plt.figure(figsize=(8, 4))
plt.hist(df['word_count'], bins=50)
plt.title('Review Length Distribution')
plt.xlabel('Words per review')
plt.ylabel('Number of reviews')
plt.savefig('plots/length_distribution.png', bbox_inches='tight')
plt.close()
results_df.to_csv('plots/results_table.csv', index=False)
print('Saved results to plots/results_table.csv')

## Dataset citation

Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). *Learning Word Vectors for Sentiment Analysis*. Proceedings of ACL-HLT 2011. Stanford Large Movie Review Dataset.

Kaggle dataset: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews